# 01 分子如何变成数据

这一节从 SMILES 开始。SMILES 是 **Simplified Molecular Input Line Entry System** 的缩写，
是一种把分子图写成一行字符串的表示方法。它适合存储、检索、放进表格，也适合作为后续
descriptor、fingerprint 和机器学习模型的起点。

应用场景：化学数据库检索、结构去重、虚拟筛选、性质预测、反应 SMILES 表示。
经典出处可追溯到 Weininger 提出的 SMILES 表示；实际使用时以 RDKit/Daylight 等工具的解析规则为准。

## 直觉解释

SMILES 可以理解为“沿着分子图走一遍，并把路线上遇到的原子、键、分支、环闭合写下来”。
因为同一个分子图可以从不同原子开始走，也可以选择不同方向和不同分支顺序，所以同一个分子
常常有多种合法 SMILES 写法。

例子：乙醇可以写成 `CCO`，也可以写成 `OCC`。它们对应同一个分子图。

为了去重和比较，我们常把输入 SMILES 转换成 canonical SMILES。canonical SMILES 的意思是：
软件先解析出分子图，再用固定的排序和遍历规则选出一种标准写法。只要分子图、同位素、
电荷、芳香性、立体化学等信息被同一个工具以同样规则理解，输出就应该是确定的。

In [ ]:
from pathlib import Path

START = Path.cwd().resolve()
for candidate in [START, *START.parents]:
    if (candidate / "data").exists() and (candidate / "notebooks").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Cannot find the materials root. Start Jupyter from the materials directory "
        "or from one of its subdirectories."
    )

DATA = ROOT / "data"
RAW = DATA / "raw"
EXAMPLES = DATA / "examples"
RANDOM_STATE = 42

print("materials root:", ROOT)

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

# 这一格先读取示例分子表。后面每个分子都会被 RDKit 解析成 molecule object。
examples = pd.read_csv(EXAMPLES / "example_molecules.csv")
examples

## 数学/化学定义

分子图可以看作：

```text
G = (V, E)
V: atoms
E: bonds
```

SMILES 是这个图的一种线性写法，不是唯一写法。canonical SMILES 是算法选择的一种标准线性写法。

需要注意：canonical SMILES 的“唯一”是实践中的唯一，不是脱离软件和化学约定的绝对唯一。
如果没有明确立体化学、质子化状态、盐形式、互变异构体或原子映射，两个字符串仍然可能代表
本教程范围之外更复杂的化学差异。

In [ ]:
def parse_status(smiles):
    # MolFromSmiles 会把字符串解析成 RDKit 分子图；解析失败时返回 None。
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return pd.Series({"valid": False, "canonical SMILES": None, "atoms": None, "bonds": None})
    return pd.Series(
        {
            "valid": True,
            "canonical SMILES": Chem.MolToSmiles(mol),
            "atoms": mol.GetNumAtoms(),
            "bonds": mol.GetNumBonds(),
        }
    )

parsed = pd.concat([examples, examples["smiles"].apply(parse_status)], axis=1)
parsed

## 代码

RDKit 可以把 SMILES 解析成分子对象，再画出二维结构。无效 SMILES 会返回 `None`。

In [ ]:
valid = parsed[parsed["valid"]].copy()
mols = [Chem.MolFromSmiles(smi) for smi in valid["smiles"]]
legends = [f"{name}\n{can}" for name, can in zip(valid["name"], valid["canonical SMILES"])]

# SVG 是矢量图，放大后仍然清晰；subImgSize 越大，legend 中的 SMILES 越容易读。
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(360, 260), legends=legends, useSVG=True)

In [ ]:
student_smiles = "CC(=O)Oc1ccccc1C(=O)O"  # try replacing this with your molecule
mol = Chem.MolFromSmiles(student_smiles)

if mol is None:
    print("RDKit cannot parse this SMILES.")
else:
    print("canonical SMILES:", Chem.MolToSmiles(mol))
    print("atoms:", mol.GetNumAtoms(), "bonds:", mol.GetNumBonds())

    mol_with_labels = Chem.Mol(mol)
    for atom in mol_with_labels.GetAtoms():
        atom.SetProp("atomNote", f"{atom.GetSymbol()}{atom.GetIdx()}")

    display(
        Draw.MolsToGridImage(
            [mol_with_labels],
            molsPerRow=1,
            subImgSize=(520, 380),
            legends=["atom labels show element symbol + atom index"],
            useSVG=True,
        )
    )

## 观察问题

1. `CCO` 和 `OCC` 的 canonical SMILES 是否相同？
2. `invalid_ring` 为什么不能被 RDKit 解析？
3. canonical SMILES 解决了什么问题？它没有解决什么问题？

### Hints

1. 先让 RDKit 分别解析 `CCO` 和 `OCC`，再比较 `Chem.MolToSmiles(...)` 的输出。
2. 无效 SMILES 常见原因包括环编号没有闭合、括号不匹配、原子价态不合理、非法字符等。
3. canonical SMILES 主要解决“同一个分子有多种字符串写法”导致的去重和比较问题；它不能自动解决
   互变异构体、质子化状态、盐形式、构象、溶剂和实验条件这些化学语境问题。

## 小结

SMILES 让分子可以进入表格和代码；canonical SMILES 让去重更容易。但 SMILES 本身不等于全部化学信息，例如实验条件、构象、溶剂和测量误差仍然需要额外记录。